In [19]:
library(tidyverse)
library(lubridate)
# Combine all HCRIS years from hwk4 data
hcris_files <- list.files(path = "../../hwk4/data/", 
                        pattern = "data-201[0-8]\\.csv", # Grabs 2010-2018
                          full.names = TRUE)

hospital_data <- hcris_files %>%
  map_df(~read_csv(., col_types = cols(
    provider_number = col_character(),
    state = col_character()
  )))

hospital_data <- hospital_data %>%
  mutate(uncomp_care_final = coalesce(uncomp_care, tot_uncomp_care_charges))

# Prepare kff data
kff.dat <- read_csv('../data/KFF.csv', skip = 2) 

kff.final <- kff.dat %>%
  rename(state_name = 1, status = 2, date_str = 3) %>%
  filter(state_name != "United States") %>%
  mutate(
    expand_ever = (status == 'Adopted'),
    expansion_year = year(mdy(date_str)),
    # Changes state names to abbreviations
    st_abb = state.abb[match(state_name, state.name)]
  ) %>%
  mutate(st_abb = ifelse(state_name == "District of Columbia", "DC", st_abb)) %>%
  select(st_abb, expand_ever, expansion_year)

# Join and calculate true costs
final_analysis_data <- hospital_data %>%
  left_join(kff.final, by = c("state" = "st_abb")) %>%
  mutate(
    universal_charges = coalesce(tot_uncomp_care_charges, uncomp_care),
    # Calculate true cost
    true_uncomp_cost = universal_charges * cost_to_charge,
    # Millions variable and treatment indicator
    uncomp_millions = true_uncomp_cost / 1000000,
    expand = (year >= expansion_year & !is.na(expansion_year))
  )

Rows: 52 Columns: 3
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): Location, Status of Medicaid Expansion Decision, Expansion Implemen...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `expansion_year = year(mdy(date_str))`.
Caused by warning:
!  10 failed to parse.”


In [21]:
write_csv(final_analysis_data, "../data/final_analysis_data.csv")